In [53]:
from ngsolve import *
from netgen.geom2d import unit_square
import numpy as np
import random
from math import sqrt

mesh = Mesh(unit_square.GenerateMesh(maxh=0.1))
fes = H1(mesh, order=1)
u,v = fes.TnT()
a = BilinearForm(grad(u)*grad(v)*dx+10*u*v*dx).Assemble()
f = LinearForm(x*y*v*dx).Assemble()
gfu = GridFunction(fes)

r = f.vec.CreateVector()
p_old = f.vec.CreateVector()
p_new = f.vec.CreateVector()
p1 = f.vec.CreateVector()

r1 = f.vec.CreateVector()
p_old1 = f.vec.CreateVector()
p2 = f.vec.CreateVector()
r1.data[:] = 0
p_old1.data[:] = 0
p2.data[:] = 0

gfu.vec[:] = 0
r.data = f.vec
p_old.data = r.data
err0 = Norm(r)

its = 0
num_samples = 40 # Number of random samples for stochastic updates
max_iters = 10000
tolerance = 1e-8

while True:
    err4 = Norm(r)
    if sqrt(err4) < tolerance or its > max_iters: 
        break
    indices = random.sample(range(len(r)), num_samples)

    r1.data[:]=0
    p_old1.data[:]=0
    for i in indices:
        r1[i] = r[i]
        p_old1[i] = p_old[i]

    p2.data = a.mat * p_old1

    err2 = InnerProduct(r1,r1)
    denom = InnerProduct(p_old1, p2)
    alpha = err2 / denom

    gfu.vec.data += alpha * p_old1
    
    r.data -= alpha * p2
    for i in indices:
        r1[i] = r[i]
    err3 = InnerProduct(r1,r1)
    if np.isnan(err3) or err3 > 1e5:
        print("Divergence detected, stopping.")
        break
    beta = err3 / err2
    if abs(beta) > 1e5:
        print("Beta exploded, stopping.")
        break
    p_new.data = r.data + beta * p_old.data
    p_old.data = p_new.data

    print(f"Iteration {its}: res={sqrt(err4):.3e}, alpha={alpha:.3e}, beta={beta:.3e}")
    
    its = its+1
print ("needed", its, "iterations")

Iteration 0: res=1.718e-01, alpha=4.342e-01, beta=7.931e-02
Iteration 1: res=1.859e-01, alpha=3.768e-01, beta=1.316e-01
Iteration 2: res=1.923e-01, alpha=3.090e-01, beta=1.224e-01
Iteration 3: res=1.923e-01, alpha=3.000e-01, beta=8.605e-02
Iteration 4: res=1.857e-01, alpha=3.621e-01, beta=1.622e-01
Iteration 5: res=1.879e-01, alpha=2.887e-01, beta=1.258e-01
Iteration 6: res=1.873e-01, alpha=2.600e-01, beta=9.334e-02
Iteration 7: res=1.864e-01, alpha=2.934e-01, beta=1.285e-01
Iteration 8: res=1.821e-01, alpha=2.978e-01, beta=1.019e-01
Iteration 9: res=1.845e-01, alpha=3.019e-01, beta=2.314e-01
Iteration 10: res=1.872e-01, alpha=2.253e-01, beta=1.235e-01
Iteration 11: res=1.805e-01, alpha=3.971e-01, beta=1.482e-01
Iteration 12: res=1.818e-01, alpha=2.948e-01, beta=1.243e-01
Iteration 13: res=1.864e-01, alpha=2.927e-01, beta=2.454e-01
Iteration 14: res=1.804e-01, alpha=3.106e-01, beta=2.667e-01
Iteration 15: res=1.848e-01, alpha=2.530e-01, beta=1.389e-01
Iteration 16: res=1.825e-01, alpha